In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

In [2]:
df=pd.read_csv('movies_metadata.csv',low_memory=False)

In [3]:
df.columns

Index(['adult', 'belongs_to_collection', 'budget', 'genres', 'homepage', 'id',
       'imdb_id', 'original_language', 'original_title', 'overview',
       'popularity', 'poster_path', 'production_companies',
       'production_countries', 'release_date', 'revenue', 'runtime',
       'spoken_languages', 'status', 'tagline', 'title', 'video',
       'vote_average', 'vote_count'],
      dtype='object')

In [4]:
df=df[['title','genres','overview','spoken_languages','tagline','popularity','vote_average']]

In [5]:
df.iloc[0]

title                                                       Toy Story
genres              [{'id': 16, 'name': 'Animation'}, {'id': 35, '...
overview            Led by Woody, Andy's toys live happily in his ...
spoken_languages             [{'iso_639_1': 'en', 'name': 'English'}]
tagline                                                           NaN
popularity                                                  21.946943
vote_average                                                      7.7
Name: 0, dtype: object

In [6]:
df.isna().sum()

title                   6
genres                  0
overview              954
spoken_languages        6
tagline             25054
popularity              5
vote_average            6
dtype: int64

In [7]:
df=df.dropna(subset=['title']).reset_index(drop=True)

In [8]:
df['overview']=df['overview'].fillna(' ')
df['tagline']=df['tagline'].fillna(' ')

In [9]:
df.duplicated().sum()

np.int64(17)

In [10]:
df.drop_duplicates(inplace=True)

In [11]:
df['genres']

0        [{'id': 16, 'name': 'Animation'}, {'id': 35, '...
1        [{'id': 12, 'name': 'Adventure'}, {'id': 14, '...
2        [{'id': 10749, 'name': 'Romance'}, {'id': 35, ...
3        [{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...
4                           [{'id': 35, 'name': 'Comedy'}]
                               ...                        
45455    [{'id': 18, 'name': 'Drama'}, {'id': 10751, 'n...
45456                        [{'id': 18, 'name': 'Drama'}]
45457    [{'id': 28, 'name': 'Action'}, {'id': 18, 'nam...
45458                                                   []
45459                                                   []
Name: genres, Length: 45443, dtype: object

In [12]:
import ast
df['genres']=df['genres'].apply(lambda x: ' '.join([i['name'] for i in ast.literal_eval(x)]))
df['spoken_languages']=df['spoken_languages'].apply(lambda x: ' '.join([i['name'] for i in ast.literal_eval(x)]))

In [13]:
df['genres']

0         Animation Comedy Family
1        Adventure Fantasy Family
2                  Romance Comedy
3            Comedy Drama Romance
4                          Comedy
                   ...           
45455                Drama Family
45456                       Drama
45457       Action Drama Thriller
45458                            
45459                            
Name: genres, Length: 45443, dtype: object

In [14]:
df.head()

,title,genres,overview,spoken_languages,tagline,popularity,vote_average
0,Toy Story,Animation Comedy Family,"Led by Woody, Andy's toys live happily in his ...",English,,21.946943,7.7
1,Jumanji,Adventure Fantasy Family,When siblings Judy and Peter discover an encha...,English Français,Roll the dice and unleash the excitement!,17.015539,6.9
2,Grumpier Old Men,Romance Comedy,A family wedding reignites the ancient feud be...,English,Still Yelling. Still Fighting. Still Ready for...,11.7129,6.5
3,Waiting to Exhale,Comedy Drama Romance,"Cheated on, mistreated and stepped on, the wom...",English,Friends are the people who let you be yourself...,3.859495,6.1
4,Father of the Bride Part II,Comedy,Just when George Banks has recovered from his ...,English,Just When His World Is Back To Normal... He's ...,8.387519,5.7


In [15]:
df['tags']=df['genres'] +" "+ df['overview'] +" "+ df['tagline']

In [16]:
df=df.reset_index(drop=True)

In [17]:
#Converting Title into Series along with
Indices=pd.Series(df.index, index=df['title']).drop_duplicates()
Indices

title
Toy Story                          0
Jumanji                            1
Grumpier Old Men                   2
Waiting to Exhale                  3
Father of the Bride Part II        4
                               ...  
Subdue                         45438
Century of Birthing            45439
Betrayal                       45440
Satan Triumphant               45441
Queerama                       45442
Length: 45443, dtype: int64

In [18]:
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [19]:
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to C:\Users\IKRAM
[nltk_data]     RIND\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to C:\Users\IKRAM
[nltk_data]     RIND\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [20]:
stop_words=set(stopwords.words('english'))
lemmatizer=WordNetLemmatizer()

In [21]:
import re
def preprocess_text(text):
#lowercase
    text = str(text).lower()
#removed the punctuations
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    words = text.split()
#remove stopwords
    words = [word for word in words if word not in stop_words]

#lemmatize
    words = [lemmatizer.lemmatize(word) for word in words]

    return " ".join(words)
df['tags']=df['tags'].apply(preprocess_text)    

In [22]:
df['tags'][1]

'adventure fantasy family sibling judy peter discover enchanted board game open door magical world unwittingly invite alan adult who trapped inside game year living room alans hope freedom finish game prof risky three find running giant rhinoceros evil monkey terrifying creature roll dice unleash excitement'

In [23]:
df=df.reset_index(drop=True)

In [24]:
indices=pd.Series(df.index , index=df['title']).drop_duplicates()

In [25]:
indices

title
Toy Story                          0
Jumanji                            1
Grumpier Old Men                   2
Waiting to Exhale                  3
Father of the Bride Part II        4
                               ...  
Subdue                         45438
Century of Birthing            45439
Betrayal                       45440
Satan Triumphant               45441
Queerama                       45442
Length: 45443, dtype: int64

In [26]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [27]:
vector=TfidfVectorizer(max_features=50000, #50000 THousand WOrds are only for Vocabulry
                       ngram_range=(1,2),   #1 string than 2 cobine string again one string
                       stop_words='english')

In [28]:
vec_tfidf=vector.fit_transform(df['tags'])

In [29]:
vec_tfidf

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 1543928 stored elements and shape (45443, 50000)>

In [30]:
from sklearn.metrics.pairwise import cosine_similarity

In [52]:
def recomended(title , n=10):
    if title not in indices:
        return 'Movie Not Found'
    idx=indices[title]
    sim_score=cosine_similarity(vec_tfidf[idx],vec_tfidf).flatten()
    similar_idx=sim_score.argsort()[::-1][1:n+1]
    return df['title'].iloc[similar_idx]
    

In [53]:
recomended('Toy Story')

2996                Toy Story 2
15344               Toy Story 3
24512                 Small Fry
17184                 Group Sex
28972           Superstar Goofy
6434     What's Up, Tiger Lily?
11396    For Your Consideration
1071      Rebel Without a Cause
1931                  Condorman
485                      Malice
Name: title, dtype: object

In [55]:
import pickle

pickle.dump(vec_tfidf, open('vec_tfidf.pkl','wb'))

pickle.dump(indices, open('indices.pkl','wb'))

df.to_pickle('df.pkl')

pickle.dump(vector,open('tfidf.pkl','wb'))